# Dump Opsim Database
- creation : 2026-08-10
- last update : 2026-08-10
- author : Sylvie Dagoret-Campagne

## 1. Import

In [ ]:
import warnings
import logging
import urllib
from os import path
import numpy as np
import pandas as pd

from collections import OrderedDict

import matplotlib.pyplot as plt
from scipy.constants import golden
import rubin_sim.maf.db

from rubin_sim.data import get_baseline

In [ ]:
%matplotlib inline
# %config InlineBackend.figure_format = 'svg'
# %load_ext lab_black
# %load_ext pycodestyle_magic
# %flake8_on --ignore E501,W505
%load_ext autoreload
%autoreload 1

In [ ]:
warnings.filterwarnings(
    "ignore",
    append=True,
    message=r".*Tried to get polar motions for times after IERS data is valid.*",
)
warnings.filterwarnings("ignore", append=True, message=r".*dubious year.*")

## 2. Logging

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
logger = logging.getLogger("hourglass_notebook")
logger.setLevel("DEBUG")
logger.info("Starting")

## 3. Get input  database connections

In [ ]:
logger.debug("Configuring database connections")
baseline = get_baseline()

## 4. Dump

In [ ]:
# check which columns are available
import sqlite3

opsim_fname = baseline
conn = sqlite3.connect(opsim_fname)
cursor = conn.cursor()

In [ ]:
df = pd.read_sql(
    """
    SELECT *
    FROM observations
    LIMIT 1
    """,
    conn,
)

print(df.columns.tolist())

### 4.1 Table Observation dump

In [ ]:
cursor.execute("PRAGMA table_info(observations);")
cols = cursor.fetchall()

In [ ]:
list(cols)

In [ ]:
cols_to_check = ["scheduler_note", "target_name", "observation_reason", "science_program"]

for col in cols_to_check:
    try:
        print(f"\n=== {col} ===")
        # cursor.execute(f"SELECT DISTINCT {col} FROM observations;")

        cursor.execute(
            f"""
            SELECT {col}, COUNT(*) 
            FROM observations 
            GROUP BY {col}
            ORDER BY COUNT(*) DESC
            LIMIT 50;
            """
        )

        values = cursor.fetchall()
        for v in values:
            print(v[0])
    except Exception as e:
        print(f"{col} -> erreur ({e})")

In [ ]:
conn = sqlite3.connect(baseline)

for col in ["observation_reason", "target_name", "science_program"]:
    print("\n======", col, "======")
    df = pd.read_sql(
        f"""
        SELECT {col}, COUNT(*) as n
        FROM observations
        GROUP BY {col}
        ORDER BY n DESC
        LIMIT 30
        """,
        conn,
    )
    print(df)

In [ ]:
df = pd.read_sql(
    """
    SELECT scheduler_note,
           observation_reason,
           target_name
    FROM observations
    LIMIT 10000
    """,
    conn,
)

df.to_records(index=False)